# 3.22 — Decision Trees

Decision trees learn a sequence of yes/no questions that make the examples inside each child node purer than the examples in the parent. In this lesson, we build CART-style classification trees from scratch with NumPy, inspect Gini/entropy split scores, and use pruning-style validation logic so a pretty training tree is not mistaken for a durable future rule.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build decision trees one idea at a time. Run each cell in order and read the printed intermediate values — every split score, weighted average, and pruning decision is shown so the tree never feels like a black box. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, masks, sorting, and small numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any random-looking toy data.

### 1. A node is a class distribution, not a prediction yet

A decision tree starts with all training examples at the root. Before choosing any question, the root is just a bag of labels. The best single prediction at a node is the majority class, but the tree-building question is deeper: how mixed is that bag, and can one split make the children less mixed?

In [ ]:
X_w = np.array([[1.0, 0.2], [1.5, 0.4], [2.0, 1.2], [2.5, 1.4],
                [3.0, 2.0], [3.5, 2.4], [4.0, 2.7], [4.5, 3.1]])  # two numeric features.
y_w = np.array([0, 0, 0, 1, 1, 1, 1, 0])  # class labels for the root node.
counts_w = np.bincount(y_w, minlength=2)  # count examples per class.
probs_w = counts_w / counts_w.sum()  # convert counts to class fractions p_k.
print("class counts:", counts_w)
print("class probabilities:", probs_w)
assert np.allclose(probs_w, [0.5, 0.5])

▶ What you'll see: the root has four examples from each class, so its majority vote is tied.

In [ ]:
plt.figure(figsize=(4.2, 3))
plt.scatter(X_w[:, 0], X_w[:, 1], c=y_w, cmap="coolwarm", s=70, edgecolor="k")
plt.xlabel("feature 0"); plt.ylabel("feature 1")
plt.title("1: labels before any split"); plt.show()

▶ What you'll see: the classes are mostly separated by feature 0, but one class-0 point at the far right makes the problem imperfect.

*Why it's done this way:* a node prediction is only a summary of the labels that arrived there. Tree learning is therefore about changing the label distribution in child nodes, not about fitting a coefficient or distance metric.

### 2. Gini impurity measures how often a random guess would be wrong

For a node with class fractions $p_k$, CART often uses Gini impurity

$$G(t)=1-\sum_k p_{k,t}^2.$$

The term $\sum_k p_k^2$ is the probability that two independent labels drawn from the node match. Subtracting from 1 gives the probability of mismatch, so a pure node has Gini 0 and a 50/50 binary node has Gini 0.5.

In [ ]:
gini_root_w = 1.0 - np.sum(probs_w ** 2)  # 1 - (0.5^2 + 0.5^2).
print("root Gini:", round(gini_root_w, 3))
assert round(gini_root_w, 3) == 0.5

▶ What you'll see: the perfectly mixed root has Gini 0.500.

In [ ]:
p_grid_w = np.linspace(0, 1, 101)  # possible class-1 fractions in a binary node.
gini_grid_w = 1 - (p_grid_w ** 2 + (1 - p_grid_w) ** 2)  # binary Gini curve.
plt.figure(figsize=(4.4, 3))
plt.plot(p_grid_w, gini_grid_w, color="purple")
plt.scatter([0.5], [0.5], color="red")
plt.xlabel("class-1 fraction p"); plt.ylabel("Gini")
plt.title("2: Gini is largest at a 50/50 mix"); plt.show()

▶ What you'll see: impurity is 0 at pure ends and peaks at 0.5 when the node is maximally mixed.

*Why it's done this way:* squaring the class fractions rewards concentration. If one class dominates, one $p_k^2$ becomes large, the sum of squares rises, and impurity falls — exactly the behavior we want when measuring node purity.

### 3. A split is judged by weighted impurity reduction

A candidate question such as `feature 0 <= 2.75` creates left and right child nodes. The split is useful only if the children are purer **after accounting for how many examples they contain**. The CART gain is

$$\Delta=G(parent)-\sum_c \frac{n_c}{n}G(c).$$

The weights $n_c/n$ matter because a tiny pure child should not be allowed to hide a large messy child.

In [ ]:
thr_w = 2.75  # a candidate threshold on feature 0.
left_w = X_w[:, 0] <= thr_w
right_w = ~left_w
print("left labels:", y_w[left_w])
print("right labels:", y_w[right_w])
assert left_w.sum() == 4 and right_w.sum() == 4

▶ What you'll see: the left child has mostly class 0, while the right child has mostly class 1.

In [ ]:
def gini_w(labels):
    counts = np.bincount(labels, minlength=2)
    p = counts / counts.sum()
    return 1.0 - np.sum(p ** 2)

G_left_w = gini_w(y_w[left_w])
G_right_w = gini_w(y_w[right_w])
weighted_child_w = (left_w.mean() * G_left_w) + (right_w.mean() * G_right_w)
gain_w = gini_root_w - weighted_child_w
print("child Ginis:", round(G_left_w, 3), round(G_right_w, 3))
print("weighted child impurity:", round(weighted_child_w, 3))
print("impurity reduction Δ:", round(gain_w, 3))
assert round(gain_w, 3) == 0.125

▶ What you'll see: the split lowers weighted impurity from 0.500 to 0.375, so its gain is 0.125.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.scatter(X_w[:, 0], X_w[:, 1], c=y_w, cmap="coolwarm", s=70, edgecolor="k")
plt.axvline(thr_w, color="black", linestyle="--", label="x0 <= 2.75")
plt.legend(); plt.xlabel("feature 0"); plt.ylabel("feature 1")
plt.title("3: one candidate split"); plt.show()

▶ What you'll see: the vertical line separates most low-feature class-0 points from high-feature class-1 points.

*Why it's done this way:* the formula is an ERM-style local objective: choose the question that most reduces the node's average classification uncertainty. Weighting by child size keeps the score on the same scale as the parent node.

### 4. CART searches thresholds and chooses the largest gain

For numeric features, CART only needs to test thresholds between sorted unique feature values. Each threshold defines a left/right partition, we compute the same weighted impurity reduction, and the best split is the one with the largest positive gain.

In [ ]:
values_w = np.sort(np.unique(X_w[:, 0]))
thresholds_w = (values_w[:-1] + values_w[1:]) / 2  # midpoints between observed values.
print("candidate thresholds:", thresholds_w)
assert len(thresholds_w) == 7

▶ What you'll see: seven possible feature-0 thresholds for eight sorted values.

In [ ]:
gains_w = []
for t_w in thresholds_w:
    L_w = X_w[:, 0] <= t_w
    R_w = ~L_w
    child_w = L_w.mean() * gini_w(y_w[L_w]) + R_w.mean() * gini_w(y_w[R_w])
    gains_w.append(gini_root_w - child_w)
gains_w = np.array(gains_w)
best_idx_w = int(np.argmax(gains_w))
print("gains:", np.round(gains_w, 3))
print("best threshold:", thresholds_w[best_idx_w], "gain:", round(gains_w[best_idx_w], 3))
assert round(float(thresholds_w[best_idx_w]), 3) == 2.25
assert round(float(gains_w[best_idx_w]), 3) == 0.3

▶ What you'll see: the best root split is `feature 0 <= 2.25` with gain 0.300.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.plot(thresholds_w, gains_w, marker="o", color="teal")
plt.axvline(thresholds_w[best_idx_w], color="red", linestyle="--", label="best")
plt.xlabel("threshold on feature 0"); plt.ylabel("Gini gain")
plt.title("4: threshold search"); plt.legend(); plt.show()

▶ What you'll see: one threshold has the highest gain, so CART would ask that question first.

*Why it's done this way:* midpoint testing is enough because moving a threshold inside the same gap does not change which training examples go left or right. CART searches partitions, not all real numbers.

### 5. Entropy is another purity score with the same goal

Entropy uses

$$H(t)=-\sum_k p_{k,t}\log_2 p_{k,t}$$

and information gain replaces Gini gain with entropy reduction. Entropy comes from coding/information theory: a pure node needs 0 bits to identify the class, while a 50/50 binary node needs 1 bit.

In [ ]:
def entropy_w(labels):
    counts = np.bincount(labels, minlength=2)
    p = counts[counts > 0] / counts.sum()
    return float(-np.sum(p * np.log2(p)))

H_root_w = entropy_w(y_w)
print("root entropy:", round(H_root_w, 3))
assert round(H_root_w, 3) == 1.0

▶ What you'll see: the 50/50 root has entropy 1 bit.

In [ ]:
entropy_gains_w = []
for t_w in thresholds_w:
    L_w = X_w[:, 0] <= t_w
    R_w = ~L_w
    child_H_w = L_w.mean() * entropy_w(y_w[L_w]) + R_w.mean() * entropy_w(y_w[R_w])
    entropy_gains_w.append(H_root_w - child_H_w)
entropy_gains_w = np.array(entropy_gains_w)
print("entropy gains:", np.round(entropy_gains_w, 3))
print("best entropy threshold:", thresholds_w[int(np.argmax(entropy_gains_w))])
assert thresholds_w[int(np.argmax(entropy_gains_w))] == 2.25

▶ What you'll see: entropy also prefers the same threshold in this toy data.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.plot(thresholds_w, gains_w, marker="o", label="Gini gain")
plt.plot(thresholds_w, entropy_gains_w, marker="s", label="entropy gain")
plt.xlabel("threshold on feature 0"); plt.ylabel("gain")
plt.title("5: two impurity criteria"); plt.legend(); plt.show()

▶ What you'll see: the scales differ, but both criteria rank the strongest split similarly here.

*Why it's done this way:* Gini and entropy both reward concentrated child distributions. Gini is cheaper algebraically; entropy has an information interpretation. The modeling logic — maximize weighted purity improvement — is the same.

### 6. Growing recursively creates a piecewise-constant model

After the best root question, each child can be split again using the same rule. A leaf predicts the majority class among the training examples that reach it, so the final tree is a set of rectangular regions with constant predictions.

In [ ]:
root_thr_w = 2.25
root_left_w = X_w[:, 0] <= root_thr_w
right_X_w = X_w[~root_left_w]
right_y_w = y_w[~root_left_w]
print("root-left labels:", y_w[root_left_w])
print("root-right labels:", right_y_w)
assert np.all(y_w[root_left_w] == 0)

▶ What you'll see: the left child is pure class 0, while the right child still contains one class-0 outlier.

In [ ]:
right_vals_w = np.sort(np.unique(right_X_w[:, 0]))
right_thrs_w = (right_vals_w[:-1] + right_vals_w[1:]) / 2
right_parent_g_w = gini_w(right_y_w)
right_gains_w = []
for t_w in right_thrs_w:
    L_w = right_X_w[:, 0] <= t_w
    R_w = ~L_w
    child_w = (L_w.sum()/len(right_y_w))*gini_w(right_y_w[L_w]) + (R_w.sum()/len(right_y_w))*gini_w(right_y_w[R_w])
    right_gains_w.append(right_parent_g_w - child_w)
right_gains_w = np.array(right_gains_w)
print("right-child thresholds:", right_thrs_w)
print("right-child gains:", np.round(right_gains_w, 3))
assert round(float(right_parent_g_w), 3) == 0.320

▶ What you'll see: the right child has smaller possible gains because it is already mostly pure.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.scatter(X_w[:, 0], X_w[:, 1], c=y_w, cmap="coolwarm", s=70, edgecolor="k")
plt.axvline(root_thr_w, color="black", linestyle="--", label="root split")
plt.axvline(right_thrs_w[int(np.argmax(right_gains_w))], color="gray", linestyle=":", label="next split")
plt.xlabel("feature 0"); plt.ylabel("feature 1")
plt.title("6: recursive splits make regions"); plt.legend(); plt.show()

▶ What you'll see: a second split can isolate the far-right outlier, creating smaller constant-prediction regions.

*Why it's done this way:* recursion applies the same local impurity logic until a stopping rule says the node is pure enough, too small, or too deep. The tree becomes flexible by stacking simple axis-aligned questions.

### 7. Pruning and validation restrain attractive but brittle splits

A tree can keep splitting until it memorizes quirks. The lesson's selection arithmetic makes the guardrail explicit: compute a raw empirical score, add a complexity cost, compare alternatives, and prefer the lowest full decision score rather than the prettiest training fragment.

In [ ]:
losses_w = np.array([0.224, 0.122, 0.471])  # verified per-example losses from the lesson prose.
R_S_w = float(losses_w.mean())
cost_w = 0.090
score_w = R_S_w + cost_w
print("empirical risk:", round(R_S_w, 3))
print("score with cost:", round(score_w, 3))
assert round(R_S_w, 3) == 0.272
assert round(score_w, 3) == 0.362

▶ What you'll see: the raw average 0.272 becomes a selection score 0.362 after adding cost.

In [ ]:
flex_score_w = 0.406
gap_w = flex_score_w - score_w
relative_gap_w = gap_w / flex_score_w
stable_w = 0.80 * score_w
final_scores_w = np.array([score_w, flex_score_w, stable_w])
print("gap:", round(gap_w, 3), "relative gap:", round(relative_gap_w, 3))
print("stabilized score:", round(stable_w, 3))
print("winning score:", round(final_scores_w.min(), 3))
assert round(gap_w, 3) == 0.044
assert round(relative_gap_w, 3) == 0.108
assert round(stable_w, 3) == 0.290

▶ What you'll see: the stabilized version has the lowest full score, even though raw fit alone might tempt a deeper tree.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.bar(["baseline", "flexible", "stabilized"], final_scores_w, color=["steelblue", "indianred", "seagreen"])
plt.ylabel("decision score (lower is better)")
plt.title("7: pruning-style score comparison"); plt.show()

▶ What you'll see: the green stabilized bar is the one to carry forward in this verified toy comparison.

*Why it's done this way:* pruning is regularization for trees. It accepts that lower training impurity can be purchased by fragile extra splits, so model selection uses a validation/cost-adjusted score that reflects future performance risk.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses small
> numbers, prints the intermediate values, draws one picture, and ends with an `assert`.

### ✍️ Toy 1 · A node stores a label distribution

Before a split, a node is just a small bag of labels summarized by counts, fractions, and a majority vote.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_y = np.array([0, 1, 1, 0, 1, 1, 0, 1])
print("node labels:", t1_y.tolist())  # -> [0, 1, 1, 0, 1, 1, 0, 1]
t1_counts = np.bincount(t1_y, minlength=2)
print("class counts:", t1_counts.tolist())  # -> [3, 5]
t1_total = np.sum(t1_counts)
print("node size:", int(t1_total))  # -> 8
t1_probs = t1_counts / t1_total
print("class fractions:", np.round(t1_probs, 3).tolist())  # -> [0.375, 0.625]
t1_majority = int(np.argmax(t1_counts))
print("majority class:", t1_majority)  # -> 1
assert t1_majority == 1

plt.figure(figsize=(4.2, 2.8))
plt.bar(["class 0", "class 1"], t1_counts, color=["steelblue", "orange"])
plt.ylabel("examples")
plt.title("Toy 1 · node label counts")
plt.show()

▶ What you'll see: the node leans toward class 1, but it is not pure.

### ✍️ Toy 2 · Gini impurity squares class fractions

Gini impurity is low when one class fraction dominates and high when the fractions are mixed.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_counts = np.array([5, 3])
print("class counts:", t2_counts.tolist())  # -> [5, 3]
t2_probs = t2_counts / np.sum(t2_counts)
print("class fractions:", t2_probs.tolist())  # -> [0.625, 0.375]
t2_squares = t2_probs ** 2
print("squared fractions:", np.round(t2_squares, 3).tolist())  # -> [0.391, 0.141]
t2_match_prob = np.sum(t2_squares)
print("same-class probability:", round(float(t2_match_prob), 3))  # -> 0.531
t2_gini = 1.0 - t2_match_prob
print("Gini impurity:", round(float(t2_gini), 3))  # -> 0.469
assert round(float(t2_gini), 3) == 0.469

plt.figure(figsize=(4.4, 2.8))
plt.bar(["sum p²", "Gini"], [t2_match_prob, t2_gini], color=["gray", "purple"])
plt.ylim(0, 1)
plt.title("Toy 2 · Gini = 1 - sum p²")
plt.show()

▶ What you'll see: the 5-vs-3 mix has Gini `0.469`, lower than a perfect 4-vs-4 mix.

### ✍️ Toy 3 · Weighted impurity reduction scores one split

A split is valuable only when the child impurities, weighted by child sizes, are below the parent impurity.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_x = np.array([1, 2, 3, 4, 5, 6, 7, 8])
print("feature values:", t3_x.tolist())  # -> [1, 2, 3, 4, 5, 6, 7, 8]
t3_y = np.array([0, 0, 0, 1, 1, 1, 1, 0])
print("labels:", t3_y.tolist())  # -> [0, 0, 0, 1, 1, 1, 1, 0]
t3_parent_counts = np.bincount(t3_y, minlength=2)
print("parent counts:", t3_parent_counts.tolist())  # -> [4, 4]
t3_parent_probs = t3_parent_counts / np.sum(t3_parent_counts)
print("parent fractions:", t3_parent_probs.tolist())  # -> [0.5, 0.5]
t3_parent_gini = 1.0 - np.sum(t3_parent_probs ** 2)
print("parent Gini:", round(float(t3_parent_gini), 3))  # -> 0.5
t3_threshold = 3.5
print("threshold:", t3_threshold)  # -> 3.5
t3_left = t3_x <= t3_threshold
print("left mask:", t3_left.astype(int).tolist())  # -> [1, 1, 1, 0, 0, 0, 0, 0]
t3_left_counts = np.bincount(t3_y[t3_left], minlength=2)
print("left counts:", t3_left_counts.tolist())  # -> [3, 0]
t3_right_counts = np.bincount(t3_y[~t3_left], minlength=2)
print("right counts:", t3_right_counts.tolist())  # -> [1, 4]
t3_left_probs = t3_left_counts / np.sum(t3_left_counts)
print("left fractions:", t3_left_probs.tolist())  # -> [1.0, 0.0]
t3_right_probs = t3_right_counts / np.sum(t3_right_counts)
print("right fractions:", t3_right_probs.tolist())  # -> [0.2, 0.8]
t3_left_gini = 1.0 - np.sum(t3_left_probs ** 2)
print("left Gini:", round(float(t3_left_gini), 3))  # -> 0.0
t3_right_gini = 1.0 - np.sum(t3_right_probs ** 2)
print("right Gini:", round(float(t3_right_gini), 3))  # -> 0.32
t3_weighted = t3_left.mean() * t3_left_gini + (~t3_left).mean() * t3_right_gini
print("weighted child Gini:", round(float(t3_weighted), 3))  # -> 0.2
t3_gain = t3_parent_gini - t3_weighted
print("Gini gain:", round(float(t3_gain), 3))  # -> 0.3
assert round(float(t3_gain), 3) == 0.3

plt.figure(figsize=(4.8, 2.8))
plt.scatter(t3_x, t3_y, c=t3_y, cmap="coolwarm", s=70, edgecolor="k")
plt.axvline(t3_threshold, color="black", linestyle="--")
plt.yticks([0, 1])
plt.xlabel("x")
plt.ylabel("class")
plt.title("Toy 3 · one split lowers Gini")
plt.show()

▶ What you'll see: the split makes a pure left child and a mostly class-1 right child.

### ✍️ Toy 4 · CART searches threshold candidates

For numeric features, candidate thresholds are midpoints between sorted training values.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_x = np.array([1, 2, 3, 4, 5, 6, 7, 8])
print("feature values:", t4_x.tolist())  # -> [1, 2, 3, 4, 5, 6, 7, 8]
t4_y = np.array([0, 0, 0, 1, 1, 1, 0, 1])
print("labels:", t4_y.tolist())  # -> [0, 0, 0, 1, 1, 1, 0, 1]
t4_thresholds = (t4_x[:-1] + t4_x[1:]) / 2
print("candidate thresholds:", t4_thresholds.tolist())  # -> [1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5]
t4_parent_counts = np.bincount(t4_y, minlength=2)
t4_parent_probs = t4_parent_counts / np.sum(t4_parent_counts)
t4_parent_gini = 1.0 - np.sum(t4_parent_probs ** 2)
print("parent Gini:", round(float(t4_parent_gini), 3))  # -> 0.5
t4_gains = []
for t4_threshold in t4_thresholds:
    t4_left = t4_x <= t4_threshold
    t4_left_counts = np.bincount(t4_y[t4_left], minlength=2)
    t4_right_counts = np.bincount(t4_y[~t4_left], minlength=2)
    t4_left_probs = t4_left_counts / np.sum(t4_left_counts)
    t4_right_probs = t4_right_counts / np.sum(t4_right_counts)
    t4_left_gini = 1.0 - np.sum(t4_left_probs ** 2)
    t4_right_gini = 1.0 - np.sum(t4_right_probs ** 2)
    t4_child_gini = t4_left.mean() * t4_left_gini + (~t4_left).mean() * t4_right_gini
    t4_gains.append(t4_parent_gini - t4_child_gini)
t4_gains = np.array(t4_gains)
print("candidate gains:", np.round(t4_gains, 3).tolist())  # -> [0.071, 0.167, 0.3, 0.125, 0.033, 0.0, 0.071]
t4_best_index = int(np.argmax(t4_gains))
print("best index:", t4_best_index)  # -> 2
t4_best_threshold = t4_thresholds[t4_best_index]
print("best threshold:", float(t4_best_threshold))  # -> 3.5
t4_best_gain = t4_gains[t4_best_index]
print("best gain:", round(float(t4_best_gain), 3))  # -> 0.3
assert round(float(t4_best_gain), 3) == 0.3

plt.figure(figsize=(4.8, 2.8))
plt.plot(t4_thresholds, t4_gains, marker="o", color="teal")
plt.axvline(t4_best_threshold, color="crimson", linestyle="--")
plt.xlabel("threshold")
plt.ylabel("Gini gain")
plt.title("Toy 4 · threshold search")
plt.show()

▶ What you'll see: threshold `3.5` has the largest gain in this candidate menu.

### ✍️ Toy 5 · Entropy gain uses bits instead of Gini units

Entropy changes the impurity scale, but it still rewards splits that make child labels concentrated.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_x = np.array([1, 2, 3, 4, 5, 6, 7, 8])
print("feature values:", t5_x.tolist())  # -> [1, 2, 3, 4, 5, 6, 7, 8]
t5_y = np.array([0, 0, 0, 1, 1, 1, 1, 1])
print("labels:", t5_y.tolist())  # -> [0, 0, 0, 1, 1, 1, 1, 1]
t5_counts = np.bincount(t5_y, minlength=2)
print("parent counts:", t5_counts.tolist())  # -> [3, 5]
t5_probs = t5_counts[t5_counts > 0] / np.sum(t5_counts)
print("parent nonzero fractions:", t5_probs.tolist())  # -> [0.375, 0.625]
t5_parent_entropy = -np.sum(t5_probs * np.log2(t5_probs))
print("parent entropy:", round(float(t5_parent_entropy), 3))  # -> 0.954
t5_threshold = 3.5
print("threshold:", t5_threshold)  # -> 3.5
t5_left = t5_x <= t5_threshold
print("left mask:", t5_left.astype(int).tolist())  # -> [1, 1, 1, 0, 0, 0, 0, 0]
t5_left_counts = np.bincount(t5_y[t5_left], minlength=2)
print("left counts:", t5_left_counts.tolist())  # -> [3, 0]
t5_right_counts = np.bincount(t5_y[~t5_left], minlength=2)
print("right counts:", t5_right_counts.tolist())  # -> [0, 5]
t5_left_probs = t5_left_counts[t5_left_counts > 0] / np.sum(t5_left_counts)
t5_right_probs = t5_right_counts[t5_right_counts > 0] / np.sum(t5_right_counts)
t5_left_entropy = -np.sum(t5_left_probs * np.log2(t5_left_probs))
print("left entropy:", round(float(t5_left_entropy), 3))  # -> -0.0
t5_right_entropy = -np.sum(t5_right_probs * np.log2(t5_right_probs))
print("right entropy:", round(float(t5_right_entropy), 3))  # -> -0.0
t5_child_entropy = t5_left.mean() * t5_left_entropy + (~t5_left).mean() * t5_right_entropy
print("weighted child entropy:", round(float(t5_child_entropy), 3))  # -> -0.0
t5_gain = t5_parent_entropy - t5_child_entropy
print("entropy gain:", round(float(t5_gain), 3))  # -> 0.954
assert round(float(t5_gain), 3) == 0.954

plt.figure(figsize=(4.4, 2.8))
plt.bar(["parent", "children", "gain"], [t5_parent_entropy, t5_child_entropy, t5_gain], color=["gray", "teal", "green"])
plt.ylabel("bits")
plt.title("Toy 5 · entropy reduction")
plt.show()

▶ What you'll see: a perfect split sends both child entropies to zero, so all parent entropy becomes gain.

### ✍️ Toy 6 · Recursive splits create piecewise-constant regions

A deeper tree applies a second question only inside the branch that still needs refinement.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)
t6_x = np.array([1, 2, 3, 4, 5, 6, 7, 8])
print("training x:", t6_x.tolist())  # -> [1, 2, 3, 4, 5, 6, 7, 8]
t6_y = np.array([0, 0, 0, 1, 1, 1, 0, 0])
print("training labels:", t6_y.tolist())  # -> [0, 0, 0, 1, 1, 1, 0, 0]
t6_root_threshold = 3.5
print("root threshold:", t6_root_threshold)  # -> 3.5
t6_right_threshold = 6.5
print("right-branch threshold:", t6_right_threshold)  # -> 6.5
t6_test = np.array([1, 2, 4, 5, 7, 8])
print("test x:", t6_test.tolist())  # -> [1, 2, 4, 5, 7, 8]
t6_go_left = t6_test <= t6_root_threshold
print("root-left mask:", t6_go_left.astype(int).tolist())  # -> [1, 1, 0, 0, 0, 0]
t6_go_middle = (t6_test > t6_root_threshold) & (t6_test <= t6_right_threshold)
print("middle-leaf mask:", t6_go_middle.astype(int).tolist())  # -> [0, 0, 1, 1, 0, 0]
t6_predictions = np.where(t6_go_left, 0, np.where(t6_go_middle, 1, 0))
print("tree predictions:", t6_predictions.tolist())  # -> [0, 0, 1, 1, 0, 0]
assert t6_predictions.tolist() == [0, 0, 1, 1, 0, 0]

plt.figure(figsize=(4.8, 2.8))
plt.scatter(t6_x, t6_y, c=t6_y, cmap="coolwarm", s=70, edgecolor="k", label="train")
plt.step(t6_test, t6_predictions, where="mid", color="black", label="tree rule")
plt.axvline(t6_root_threshold, color="gray", linestyle="--")
plt.axvline(t6_right_threshold, color="gray", linestyle=":")
plt.yticks([0, 1])
plt.xlabel("x")
plt.ylabel("class")
plt.title("Toy 6 · recursive regions")
plt.legend()
plt.show()

▶ What you'll see: the tree predicts class 1 only in the middle interval, with constant leaves elsewhere.

### ✍️ Toy 7 · Pruning compares fit plus complexity cost

A deeper tree can reduce raw loss but still lose after a complexity penalty is added.

In [ ]:
import numpy as np

t7_rng = np.random.default_rng(0)
t7_shallow_losses = np.array([0.18, 0.10, 0.16, 0.11, 0.14, 0.13])
print("shallow validation losses:", t7_shallow_losses.tolist())  # -> [0.18, 0.1, 0.16, 0.11, 0.14, 0.13]
t7_deep_losses = np.array([0.04, 0.06, 0.05, 0.07, 0.03, 0.05])
print("deep validation losses:", t7_deep_losses.tolist())  # -> [0.04, 0.06, 0.05, 0.07, 0.03, 0.05]
t7_shallow_risk = np.mean(t7_shallow_losses)
print("shallow raw risk:", round(float(t7_shallow_risk), 3))  # -> 0.137
t7_deep_risk = np.mean(t7_deep_losses)
print("deep raw risk:", round(float(t7_deep_risk), 3))  # -> 0.05
t7_shallow_cost = 0.08
print("shallow cost:", t7_shallow_cost)  # -> 0.08
t7_deep_cost = 0.20
print("deep cost:", t7_deep_cost)  # -> 0.2
t7_shallow_score = t7_shallow_risk + t7_shallow_cost
print("shallow full score:", round(float(t7_shallow_score), 3))  # -> 0.217
t7_deep_score = t7_deep_risk + t7_deep_cost
print("deep full score:", round(float(t7_deep_score), 3))  # -> 0.25
t7_scores = np.array([t7_shallow_score, t7_deep_score])
print("scores:", np.round(t7_scores, 3).tolist())  # -> [0.217, 0.25]
assert t7_shallow_score < t7_deep_score

plt.figure(figsize=(4.4, 2.8))
plt.bar(["shallow", "deep"], t7_scores, color=["seagreen", "indianred"])
plt.ylabel("risk + cost")
plt.title("Toy 7 · pruning score")
plt.show()

▶ What you'll see: the deep tree has lower raw risk, but the shallow tree has the lower full score.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, masks, sorting, impurity calculations, and numerical assertions.
import matplotlib.pyplot as plt # load Matplotlib so each split, curve, and tree diagnostic can be inspected visually.
np.random.seed(0) # make all examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Count labels in a node

**Goal.** Turn labels into class counts and fractions, because every decision-tree split starts by measuring the class mixture inside the current node. We build it in 2 steps.

In [ ]:
y_b1 = np.array([0, 0, 0, 1, 1, 1, 1, 0]) # store the labels currently sitting in one node.
counts_b1 = np.bincount(y_b1, minlength=2) # count how many examples belong to each class.
print("counts:", counts_b1) # inspect the raw node composition.
assert counts_b1.tolist() == [4, 4]

▶ What you'll see: the node contains four class-0 and four class-1 examples.

In [ ]:
probs_b1 = counts_b1 / counts_b1.sum() # convert counts to fractions p_k for impurity formulas.
print("fractions:", probs_b1) # inspect the normalized class distribution.
plt.figure(figsize=(4, 3)) # create a compact class-count plot.
plt.bar(["class 0", "class 1"], counts_b1, color=["steelblue", "orange"]) # visualize the node mixture.
plt.title("Basic 1: node label counts") # title the plot.
plt.ylabel("examples") # label the count axis.
plt.show() # display the bar chart.

▶ What you'll see: both bars are equal, so the node is maximally mixed for two classes.

👀 Takeaway: a tree node is summarized by the label distribution of examples that reach it.

### Basic 2 — Compute Gini impurity

**Goal.** Compute $1-\sum_k p_k^2$, because CART needs a numeric purity score before it can compare splits. We build it in 2 steps.

In [ ]:
probs_b2 = np.array([0.5, 0.5]) # use the balanced node fractions from Basic 1.
squares_b2 = probs_b2 ** 2 # square each class fraction for the Gini formula.
print("squared fractions:", squares_b2) # inspect the match-probability pieces.

▶ What you'll see: each class contributes 0.25 to the same-class probability.

In [ ]:
gini_b2 = 1.0 - np.sum(squares_b2) # subtract the same-class probability from 1.
print("Gini impurity:", round(gini_b2, 3)) # inspect the final impurity.
assert round(gini_b2, 3) == 0.5
plt.figure(figsize=(4, 3)) # create a compact formula breakdown plot.
plt.bar(["sum p²", "Gini"], [np.sum(squares_b2), gini_b2], color=["gray", "purple"]) # compare purity and impurity pieces.
plt.title("Basic 2: Gini = 1 - sum p²") # title the plot.
plt.show() # display the formula components.

▶ What you'll see: a balanced binary node has Gini 0.5.

👀 Takeaway: lower Gini means a node is closer to one-class purity.

### Basic 3 — Compute entropy impurity

**Goal.** Compute entropy in bits, because some trees use information gain instead of Gini gain. We build it in 2 steps.

In [ ]:
probs_b3 = np.array([0.5, 0.5]) # define a balanced binary node.
terms_b3 = -probs_b3 * np.log2(probs_b3) # compute one entropy contribution per class.
print("entropy terms:", terms_b3) # inspect the per-class bit contributions.

▶ What you'll see: each class contributes 0.5 bits.

In [ ]:
entropy_b3 = float(np.sum(terms_b3)) # add class contributions to get node entropy.
print("entropy:", round(entropy_b3, 3)) # inspect the impurity in bits.
assert round(entropy_b3, 3) == 1.0
plt.figure(figsize=(4, 3)) # create a compact entropy plot.
plt.bar(["class 0", "class 1"], terms_b3, color="teal") # show contribution by class.
plt.title("Basic 3: entropy contributions") # title the plot.
plt.ylabel("bits") # label the bit scale.
plt.show() # display the bar chart.

▶ What you'll see: a 50/50 node needs 1 bit to identify the class.

👀 Takeaway: entropy and Gini use different scales but both measure label mixture.

### Basic 4 — Make one threshold split

**Goal.** Apply one yes/no question to a feature, because numeric decision trees split rows by threshold comparisons. We build it in 2 steps.

In [ ]:
x_b4 = np.array([1.0, 1.5, 2.0, 2.5, 3.0, 3.5]) # define one numeric feature column.
y_b4 = np.array([0, 0, 0, 1, 1, 1]) # define labels aligned with the feature values.
thr_b4 = 2.25 # choose one candidate threshold.
left_b4 = x_b4 <= thr_b4 # route examples left if the question is true.
print("left indices:", np.where(left_b4)[0]) # inspect which rows go left.

▶ What you'll see: the first three rows go to the left child.

In [ ]:
print("left labels:", y_b4[left_b4], "right labels:", y_b4[~left_b4]) # inspect child label bags.
plt.figure(figsize=(4, 3)) # create a one-dimensional split visualization.
plt.scatter(x_b4, np.zeros_like(x_b4), c=y_b4, cmap="coolwarm", s=80, edgecolor="k") # draw examples on a line.
plt.axvline(thr_b4, color="black", linestyle="--") # draw the threshold question.
plt.yticks([]); plt.title("Basic 4: x <= 2.25 split"); plt.show() # display the split.
assert np.all(y_b4[left_b4] == 0)

▶ What you'll see: the threshold perfectly separates this tiny ordered example.

👀 Takeaway: a threshold split is just a boolean mask that partitions the training rows.

### Basic 5 — Weight child impurities

**Goal.** Average child impurities by child size, because a split's score must represent all examples in the parent node. We build it in 2 steps.

In [ ]:
y_left_b5 = np.array([0, 0, 0, 1]) # define labels in a left child.
y_right_b5 = np.array([1, 1, 1, 0]) # define labels in a right child.
def gini_b5(labels):
    counts = np.bincount(labels, minlength=2)
    p = counts / counts.sum()
    return 1 - np.sum(p ** 2)
print("child Ginis:", round(gini_b5(y_left_b5), 3), round(gini_b5(y_right_b5), 3)) # inspect each child impurity.

▶ What you'll see: both 3-vs-1 children have Gini 0.375.

In [ ]:
n_left_b5 = len(y_left_b5) # count left child examples.
n_right_b5 = len(y_right_b5) # count right child examples.
weighted_b5 = (n_left_b5 * gini_b5(y_left_b5) + n_right_b5 * gini_b5(y_right_b5)) / (n_left_b5 + n_right_b5) # compute weighted average.
print("weighted child impurity:", round(weighted_b5, 3)) # inspect the split's post-question impurity.
assert round(weighted_b5, 3) == 0.375
plt.figure(figsize=(4, 3)) # create a compact weighting plot.
plt.bar(["left", "right", "weighted"], [gini_b5(y_left_b5), gini_b5(y_right_b5), weighted_b5], color="slateblue") # compare components.
plt.title("Basic 5: child impurity average") # title the plot.
plt.show() # display the chart.

▶ What you'll see: equal-size children make the weighted average equal to either child impurity.

👀 Takeaway: split quality is based on the weighted child impurity, not just the purest child.

### Basic 6 — Compute impurity reduction

**Goal.** Subtract weighted child impurity from parent impurity, because CART chooses splits with large positive reduction. We build it in 2 steps.

In [ ]:
parent_gini_b6 = 0.5 # use a balanced binary parent.
child_weighted_b6 = 0.375 # use the weighted child impurity from Basic 5.
gain_b6 = parent_gini_b6 - child_weighted_b6 # compute impurity reduction.
print("gain:", round(gain_b6, 3)) # inspect how much the split helped.
assert round(gain_b6, 3) == 0.125

▶ What you'll see: the split improves impurity by 0.125.

In [ ]:
plt.figure(figsize=(4, 3)) # create a before-after impurity plot.
plt.bar(["parent", "children", "gain"], [parent_gini_b6, child_weighted_b6, gain_b6], color=["gray", "teal", "green"]) # show improvement arithmetic.
plt.title("Basic 6: impurity reduction") # title the plot.
plt.ylabel("Gini units") # label the impurity scale.
plt.show() # display the bar chart.

▶ What you'll see: the gain is exactly the vertical drop from parent impurity to weighted child impurity.

👀 Takeaway: a useful split makes the weighted children purer than the parent.

### Basic 7 — Generate candidate thresholds

**Goal.** Build midpoint thresholds from sorted feature values, because only partitions between observed values can change training membership. We build it in 2 steps.

In [ ]:
x_b7 = np.array([1.0, 1.5, 2.0, 2.5, 3.0]) # define sorted feature values.
unique_b7 = np.unique(x_b7) # remove duplicate values before creating thresholds.
print("unique values:", unique_b7) # inspect possible cut locations.

▶ What you'll see: five ordered feature values.

In [ ]:
thresholds_b7 = (unique_b7[:-1] + unique_b7[1:]) / 2 # compute midpoints between adjacent values.
print("thresholds:", thresholds_b7) # inspect candidate thresholds.
assert np.allclose(thresholds_b7, [1.25, 1.75, 2.25, 2.75])
plt.figure(figsize=(4, 2.6)) # create a threshold-location plot.
plt.scatter(unique_b7, np.zeros_like(unique_b7), color="black") # draw observed values.
for t_b7 in thresholds_b7:
    plt.axvline(t_b7, color="orange", alpha=0.6) # draw candidate cuts.
plt.yticks([]); plt.title("Basic 7: midpoint candidates"); plt.show() # display candidates.

▶ What you'll see: thresholds sit in the gaps, not directly on top of training points.

👀 Takeaway: numeric CART searches the finite set of distinct train-set partitions.

### Basic 8 — Pick the best split from candidates

**Goal.** Score several thresholds and choose the maximum gain, because greedy tree building makes one local split decision at a time. We build it in 3 steps.

In [ ]:
x_b8 = np.array([1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5]) # define one feature column.
y_b8 = np.array([0, 0, 0, 1, 1, 1, 1, 0]) # define labels with one far-right outlier.
ths_b8 = (x_b8[:-1] + x_b8[1:]) / 2 # create candidate thresholds.
print("candidates:", ths_b8) # inspect the thresholds being scored.

▶ What you'll see: seven candidate splits.

In [ ]:
def gini_b8(labels):
    c = np.bincount(labels, minlength=2)
    p = c / c.sum()
    return 1 - np.sum(p ** 2)
parent_b8 = gini_b8(y_b8) # compute parent impurity once.
gains_b8 = []
for t_b8 in ths_b8:
    L_b8 = x_b8 <= t_b8
    child_b8 = L_b8.mean() * gini_b8(y_b8[L_b8]) + (~L_b8).mean() * gini_b8(y_b8[~L_b8])
    gains_b8.append(parent_b8 - child_b8)
gains_b8 = np.array(gains_b8)
print("gains:", np.round(gains_b8, 3)) # inspect every split score.

▶ What you'll see: one threshold gives the largest impurity reduction.

In [ ]:
best_b8 = int(np.argmax(gains_b8)) # find the highest-gain candidate.
print("best threshold:", ths_b8[best_b8], "best gain:", round(gains_b8[best_b8], 3)) # inspect the selected split.
assert round(float(ths_b8[best_b8]), 3) == 2.25
assert round(float(gains_b8[best_b8]), 3) == 0.3
plt.figure(figsize=(4.8, 3)) # create a gain curve.
plt.plot(ths_b8, gains_b8, marker="o", color="teal") # plot gain by threshold.
plt.axvline(ths_b8[best_b8], color="red", linestyle="--") # mark the winner.
plt.title("Basic 8: choose max gain"); plt.xlabel("threshold"); plt.ylabel("Gini gain"); plt.show() # display the search.

▶ What you'll see: the best split is the peak of the gain curve.

👀 Takeaway: CART is greedy: at each node, it picks the split with the best immediate impurity reduction.

### Basic 9 — Predict with a stump

**Goal.** Use one split plus two leaf majorities to classify new points, because a depth-1 tree is the smallest usable tree. We build it in 2 steps.

In [ ]:
thr_b9 = 2.25 # use the best split from Basic 8.
left_class_b9 = 0 # the left child is pure class 0.
right_class_b9 = 1 # the right child majority is class 1.
x_new_b9 = np.array([1.2, 2.8, 4.7]) # define new feature values to classify.
print("new x values:", x_new_b9) # inspect test inputs.

▶ What you'll see: one point falls left of the threshold and two fall right.

In [ ]:
pred_b9 = np.where(x_new_b9 <= thr_b9, left_class_b9, right_class_b9) # route each point to a leaf prediction.
print("stump predictions:", pred_b9) # inspect predicted classes.
assert pred_b9.tolist() == [0, 1, 1]
plt.figure(figsize=(4, 2.8)) # create a stump prediction plot.
plt.scatter(x_new_b9, pred_b9, c=pred_b9, cmap="coolwarm", s=90, edgecolor="k") # draw predicted classes by x.
plt.axvline(thr_b9, color="black", linestyle="--") # show the decision boundary.
plt.yticks([0, 1]); plt.title("Basic 9: stump predictions"); plt.show() # display predictions.

▶ What you'll see: the threshold turns a continuous feature into two constant prediction regions.

👀 Takeaway: a tree predicts by routing a row through questions until it lands in a leaf.

### Basic 10 — Add a complexity cost

**Goal.** Add a cost to the raw empirical score, because model selection should penalize overly flexible trees. We build it in 2 steps.

In [ ]:
losses_b10 = np.array([0.224, 0.122, 0.471]) # use the verified toy losses from the lesson block.
risk_b10 = losses_b10.mean() # compute empirical risk as an average.
cost_b10 = 0.090 # define the complexity or operational cost.
print("risk:", round(risk_b10, 3), "cost:", round(cost_b10, 3)) # inspect the two score components.
assert round(risk_b10, 3) == 0.272

▶ What you'll see: the raw average is 0.272 before cost.

In [ ]:
score_b10 = risk_b10 + cost_b10 # combine raw fit and cost.
print("selection score:", round(score_b10, 3)) # inspect the full decision score.
assert round(score_b10, 3) == 0.362
plt.figure(figsize=(4, 3)) # create a score breakdown chart.
plt.bar(["risk", "cost", "total"], [risk_b10, cost_b10, score_b10], color=["steelblue", "orange", "green"]) # show components and total.
plt.title("Basic 10: cost-adjusted score") # title the plot.
plt.ylabel("score") # label the score axis.
plt.show() # display the bar chart.

▶ What you'll see: the full score is higher than the training fragment because it includes model cost.

👀 Takeaway: pruning-style decisions compare full scores, not raw training impurity alone.

## 🟡 Easy

### Easy 1 — Fit the best decision stump

**Goal.** Implement a full best-stump search over one feature, because it is the core split-selection loop used inside every tree node. We build it in 3 steps.

In [ ]:
x_e1 = np.array([1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5]) # one numeric feature.
y_e1 = np.array([0, 0, 0, 1, 1, 1, 1, 0]) # labels with a right-side outlier.
thresholds_e1 = (np.unique(x_e1)[:-1] + np.unique(x_e1)[1:]) / 2 # candidate midpoint thresholds.
print("threshold count:", len(thresholds_e1)) # inspect how many splits will be evaluated.

▶ What you'll see: seven candidate splits.

In [ ]:
def gini_e1(labels):
    counts = np.bincount(labels, minlength=2)
    p = counts / counts.sum()
    return 1 - np.sum(p ** 2)
parent_e1 = gini_e1(y_e1) # compute root impurity.
scores_e1 = [] # store one gain per candidate.
for t_e1 in thresholds_e1:
    L_e1 = x_e1 <= t_e1
    weighted_e1 = L_e1.mean() * gini_e1(y_e1[L_e1]) + (~L_e1).mean() * gini_e1(y_e1[~L_e1])
    scores_e1.append(parent_e1 - weighted_e1)
scores_e1 = np.array(scores_e1)
print("scores:", np.round(scores_e1, 3)) # inspect the gain table.

▶ What you'll see: each threshold's impurity reduction is printed.

In [ ]:
best_i_e1 = int(np.argmax(scores_e1)) # select the largest gain.
best_thr_e1 = float(thresholds_e1[best_i_e1]) # read the chosen threshold.
print("best stump: x <=", best_thr_e1, "gain", round(scores_e1[best_i_e1], 3)) # inspect the fitted stump.
assert round(best_thr_e1, 3) == 2.25
plt.figure(figsize=(4.8, 3)) # create a fitted-stump plot.
plt.scatter(x_e1, y_e1, c=y_e1, cmap="coolwarm", s=70, edgecolor="k") # draw training labels.
plt.axvline(best_thr_e1, color="black", linestyle="--") # show the selected threshold.
plt.title("Easy 1: fitted decision stump"); plt.xlabel("x"); plt.ylabel("class"); plt.show() # display result.

▶ What you'll see: the chosen split isolates three class-0 points on the left and a mostly class-1 region on the right.

👀 Takeaway: best-stump fitting is a finite search over candidate thresholds and impurity gains.

### Easy 2 — Search both features

**Goal.** Compare splits across two numeric features, because real CART chooses both a feature and a threshold. We build it in 3 steps.

In [ ]:
X_e2 = np.array([[1.0, 0.2], [1.5, 0.4], [2.0, 1.2], [2.5, 1.4],
                 [3.0, 2.0], [3.5, 2.4], [4.0, 2.7], [4.5, 3.1]]) # two-feature data.
y_e2 = np.array([0, 0, 0, 1, 1, 1, 1, 0]) # binary labels.
print("X shape:", X_e2.shape) # inspect rows and feature count.

▶ What you'll see: eight examples with two candidate split dimensions.

In [ ]:
def gini_e2(labels):
    c = np.bincount(labels, minlength=2)
    p = c / c.sum()
    return 1 - np.sum(p ** 2)
parent_e2 = gini_e2(y_e2) # root impurity.
best_e2 = (-1.0, None, None) # store gain, feature, threshold.
for f_e2 in range(X_e2.shape[1]):
    vals_e2 = np.sort(np.unique(X_e2[:, f_e2]))
    for t_e2 in (vals_e2[:-1] + vals_e2[1:]) / 2:
        L_e2 = X_e2[:, f_e2] <= t_e2
        child_e2 = L_e2.mean() * gini_e2(y_e2[L_e2]) + (~L_e2).mean() * gini_e2(y_e2[~L_e2])
        gain_e2 = parent_e2 - child_e2
        if gain_e2 > best_e2[0]:
            best_e2 = (gain_e2, f_e2, t_e2)
print("best gain, feature, threshold:", round(best_e2[0], 3), best_e2[1], best_e2[2]) # inspect global best split.
assert best_e2[1] == 0 and round(best_e2[2], 3) == 2.25

▶ What you'll see: feature 0 at threshold 2.25 is the best split across both columns.

In [ ]:
plt.figure(figsize=(4.4, 3)) # create a two-feature visualization.
plt.scatter(X_e2[:, 0], X_e2[:, 1], c=y_e2, cmap="coolwarm", s=70, edgecolor="k") # draw labeled points.
plt.axvline(best_e2[2], color="black", linestyle="--") # show feature-0 split.
plt.xlabel("feature 0"); plt.ylabel("feature 1")
plt.title("Easy 2: best feature-threshold pair"); plt.show() # display best split.

▶ What you'll see: the best split is vertical because feature 0 separates the classes better than feature 1 here.

👀 Takeaway: CART's split search loops over features and thresholds, then keeps the highest gain.

### Easy 3 — Build a depth-2 tree manually

**Goal.** Add a second split to the impure branch, because recursive splitting makes trees more flexible than stumps. We build it in 3 steps.

In [ ]:
x_e3 = np.array([1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5]) # one feature.
y_e3 = np.array([0, 0, 0, 1, 1, 1, 1, 0]) # labels.
root_thr_e3 = 2.25 # best root split.
right_e3 = x_e3 > root_thr_e3 # select the impure right child.
print("right child labels:", y_e3[right_e3]) # inspect what remains to split.

▶ What you'll see: the right child is mostly class 1 but contains one class-0 outlier.

In [ ]:
right_x_e3 = x_e3[right_e3] # feature values inside the right child.
right_y_e3 = y_e3[right_e3] # labels inside the right child.
right_thr_e3 = 4.25 # split between x=4.0 and the far-right outlier x=4.5.
left_leaf_e3 = 0 # root-left pure class.
mid_leaf_e3 = 1 # right-left majority class.
far_leaf_e3 = 0 # right-right isolated outlier class.
print("second split threshold:", right_thr_e3) # inspect the depth-2 question.

▶ What you'll see: the second question isolates the last point.

In [ ]:
x_test_e3 = np.array([1.4, 3.2, 4.6]) # define new values to route through the depth-2 tree.
pred_e3 = np.where(x_test_e3 <= root_thr_e3, left_leaf_e3, np.where(x_test_e3 <= right_thr_e3, mid_leaf_e3, far_leaf_e3)) # route through root and second split.
print("depth-2 predictions:", pred_e3) # inspect tree outputs.
assert pred_e3.tolist() == [0, 1, 0]
plt.figure(figsize=(4.6, 2.8)) # create a region plot.
plt.scatter(x_e3, y_e3, c=y_e3, cmap="coolwarm", s=70, edgecolor="k") # draw training labels.
plt.axvline(root_thr_e3, color="black", linestyle="--") # root split.
plt.axvline(right_thr_e3, color="gray", linestyle=":") # second split.
plt.title("Easy 3: depth-2 regions"); plt.xlabel("x"); plt.ylabel("class"); plt.show() # display tree regions.

▶ What you'll see: three constant prediction intervals: class 0, class 1, then class 0.

👀 Takeaway: deeper trees create more regions, which can fit real structure or memorize outliers.

### Easy 4 — Compare train and validation error

**Goal.** Measure whether a deeper tree's training improvement survives validation, because generalization matters more than memorization. We build it in 3 steps.

In [ ]:
x_train_e4 = np.array([1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5]) # training feature.
y_train_e4 = np.array([0, 0, 0, 1, 1, 1, 1, 0]) # training labels.
x_val_e4 = np.array([1.25, 2.75, 4.6]) # validation feature values.
y_val_e4 = np.array([0, 1, 1]) # future labels where the far-right region is class 1.
print("validation labels:", y_val_e4) # inspect the future check.

▶ What you'll see: validation disagrees with the training outlier at the far right.

In [ ]:
stump_pred_train_e4 = np.where(x_train_e4 <= 2.25, 0, 1) # depth-1 predictions.
deep_pred_train_e4 = np.where(x_train_e4 <= 2.25, 0, np.where(x_train_e4 <= 4.25, 1, 0)) # depth-2 predictions.
stump_train_err_e4 = np.mean(stump_pred_train_e4 != y_train_e4) # training error for stump.
deep_train_err_e4 = np.mean(deep_pred_train_e4 != y_train_e4) # training error for deeper tree.
print("train errors stump/deep:", stump_train_err_e4, deep_train_err_e4) # inspect apparent fit.
assert deep_train_err_e4 < stump_train_err_e4

▶ What you'll see: the deeper tree has lower training error because it memorizes the outlier.

In [ ]:
stump_val_e4 = np.where(x_val_e4 <= 2.25, 0, 1) # validation predictions for stump.
deep_val_e4 = np.where(x_val_e4 <= 2.25, 0, np.where(x_val_e4 <= 4.25, 1, 0)) # validation predictions for deeper tree.
stump_val_err_e4 = np.mean(stump_val_e4 != y_val_e4) # validation error for stump.
deep_val_err_e4 = np.mean(deep_val_e4 != y_val_e4) # validation error for deeper tree.
print("validation errors stump/deep:", stump_val_err_e4, deep_val_err_e4) # inspect future behavior.
assert stump_val_err_e4 < deep_val_err_e4
plt.figure(figsize=(4, 3)) # create a train-validation comparison.
plt.bar(["stump train", "deep train", "stump val", "deep val"], [stump_train_err_e4, deep_train_err_e4, stump_val_err_e4, deep_val_err_e4], color=["steelblue", "orange", "steelblue", "orange"]) # compare errors.
plt.xticks(rotation=25); plt.ylabel("error rate"); plt.title("Easy 4: validation catches overfit"); plt.show() # display comparison.

▶ What you'll see: the deeper tree wins on training but loses on validation.

👀 Takeaway: validation error is the practical test of whether extra splits are reusable structure.

### Easy 5 — Choose with a pruning-style score

**Goal.** Combine empirical risk, cost, and a stabilized score, because pruning uses the full selection score rather than raw fit alone. We build it in 3 steps.

In [ ]:
losses_e5 = np.array([0.224, 0.122, 0.471]) # verified per-example toy losses.
risk_e5 = float(losses_e5.mean()) # empirical risk.
cost_e5 = 0.090 # complexity cost.
base_score_e5 = risk_e5 + cost_e5 # cost-adjusted score.
print("risk:", round(risk_e5, 3), "base score:", round(base_score_e5, 3)) # inspect score components.
assert round(base_score_e5, 3) == 0.362

▶ What you'll see: the baseline full score is 0.362.

In [ ]:
flex_score_e5 = 0.406 # more flexible alternative's full decision score.
stable_score_e5 = 0.80 * base_score_e5 # stabilizing knob reduces baseline score by 20%.
scores_e5 = np.array([base_score_e5, flex_score_e5, stable_score_e5]) # gather alternatives.
labels_e5 = np.array(["baseline", "flexible", "stabilized"]) # labels for ranking.
winner_e5 = labels_e5[int(np.argmin(scores_e5))] # choose the lowest score.
print("scores:", np.round(scores_e5, 3), "winner:", winner_e5) # inspect the final selection.
assert winner_e5 == "stabilized"

▶ What you'll see: the stabilized alternative has the lowest score.

In [ ]:
gap_e5 = flex_score_e5 - base_score_e5 # raw score gap between flexible and baseline.
relative_gap_e5 = gap_e5 / flex_score_e5 # relative gap on the flexible score's scale.
print("gap:", round(gap_e5, 3), "relative gap:", round(relative_gap_e5, 3)) # inspect evidence size.
assert round(gap_e5, 3) == 0.044
plt.figure(figsize=(4.8, 3)) # create final decision plot.
plt.bar(labels_e5, scores_e5, color=["steelblue", "indianred", "seagreen"]) # compare alternatives.
plt.ylabel("score lower is better"); plt.title("Easy 5: full score wins"); plt.show() # display ranking.

▶ What you'll see: score comparison, not raw training loss, chooses the tree to carry forward.

👀 Takeaway: cost and validation-like gaps are guardrails against brittle tree flexibility.

## 🔴 Advanced

### Advanced 1 — Build a small recursive tree function

**Goal.** Write a tiny recursive CART classifier, because recursion is what turns one split search into a tree. We build it in 4 steps.

In [ ]:
X_a1 = np.array([[1.0, 0.2], [1.5, 0.4], [2.0, 1.2], [2.5, 1.4],
                 [3.0, 2.0], [3.5, 2.4], [4.0, 2.7], [4.5, 3.1]]) # training features.
y_a1 = np.array([0, 0, 0, 1, 1, 1, 1, 0]) # training labels.
print("training rows:", len(y_a1)) # inspect dataset size before recursion.

▶ What you'll see: eight labeled examples.

In [ ]:
def gini_a1(labels):
    c = np.bincount(labels, minlength=2)
    p = c / c.sum()
    return 1 - np.sum(p ** 2)

def best_split_a1(X, y):
    parent = gini_a1(y)
    best = (0.0, None, None)
    for f in range(X.shape[1]):
        vals = np.sort(np.unique(X[:, f]))
        for t in (vals[:-1] + vals[1:]) / 2:
            L = X[:, f] <= t
            if L.sum() == 0 or (~L).sum() == 0:
                continue
            child = (L.sum()/len(y))*gini_a1(y[L]) + ((~L).sum()/len(y))*gini_a1(y[~L])
            gain = parent - child
            if gain > best[0]:
                best = (gain, f, t)
    return best
print("root split:", best_split_a1(X_a1, y_a1)) # inspect the root decision.

▶ What you'll see: the root split chooses feature 0 at threshold 2.25.

In [ ]:
def build_a1(X, y, depth=0, max_depth=2, min_leaf=1):
    counts = np.bincount(y, minlength=2)
    pred = int(np.argmax(counts))
    if depth == max_depth or np.max(counts) == len(y) or len(y) <= min_leaf:
        return {"leaf": True, "pred": pred, "n": len(y)}
    gain, f, t = best_split_a1(X, y)
    if gain <= 0:
        return {"leaf": True, "pred": pred, "n": len(y)}
    L = X[:, f] <= t
    return {"leaf": False, "pred": pred, "feature": f, "threshold": t, "gain": gain,
            "left": build_a1(X[L], y[L], depth + 1, max_depth, min_leaf),
            "right": build_a1(X[~L], y[~L], depth + 1, max_depth, min_leaf)}
tree_a1 = build_a1(X_a1, y_a1, max_depth=2)
print("tree root:", {k: tree_a1[k] for k in ["feature", "threshold", "gain"]}) # inspect learned root.
assert round(tree_a1["threshold"], 3) == 2.25

▶ What you'll see: a dictionary representation of the learned tree root.

In [ ]:
def predict_one_a1(node, row):
    while not node["leaf"]:
        node = node["left"] if row[node["feature"]] <= node["threshold"] else node["right"]
    return node["pred"]
preds_a1 = np.array([predict_one_a1(tree_a1, row) for row in X_a1]) # predict all training rows.
print("training predictions:", preds_a1) # inspect fitted labels.
print("training error:", np.mean(preds_a1 != y_a1)) # inspect apparent fit.
plt.figure(figsize=(4.4, 3)) # create fitted prediction plot.
plt.scatter(X_a1[:, 0], X_a1[:, 1], c=preds_a1, cmap="coolwarm", s=75, edgecolor="k") # show model predictions.
plt.title("Advanced 1: recursive tree predictions"); plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.show() # display predictions.

▶ What you'll see: the depth-2 tree fits the training set very closely by isolating the outlier.

👀 Takeaway: a full tree is a recursive repetition of best-split search plus leaf-majority prediction.

### Advanced 2 — Sweep maximum depth

**Goal.** Compare train and validation error across depths, because maximum depth is a regularization knob for trees. We build it in 4 steps.

In [ ]:
x_a2 = np.array([1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5]) # training feature.
y_a2 = np.array([0, 0, 0, 1, 1, 1, 1, 0]) # training labels.
xv_a2 = np.array([1.25, 2.75, 3.8, 4.6]) # validation feature.
yv_a2 = np.array([0, 1, 1, 1]) # validation labels.
depths_a2 = np.array([0, 1, 2]) # root leaf, stump, and depth-2 tree.
print("depths:", depths_a2) # inspect the regularization grid.

▶ What you'll see: three model capacities will be compared.

In [ ]:
def pred_depth_a2(x, depth):
    if depth == 0:
        return np.ones_like(x, dtype=int) # root majority class is class 1 by tie-breaking here.
    if depth == 1:
        return np.where(x <= 2.25, 0, 1)
    return np.where(x <= 2.25, 0, np.where(x <= 4.25, 1, 0))
train_err_a2 = []
val_err_a2 = []
for d_a2 in depths_a2:
    train_err_a2.append(np.mean(pred_depth_a2(x_a2, d_a2) != y_a2))
    val_err_a2.append(np.mean(pred_depth_a2(xv_a2, d_a2) != yv_a2))
print("train errors:", train_err_a2) # inspect fit by depth.
print("validation errors:", val_err_a2) # inspect future behavior by depth.

▶ What you'll see: training error falls with depth, while validation can prefer the simpler stump.

In [ ]:
best_depth_a2 = int(depths_a2[int(np.argmin(val_err_a2))]) # choose depth by validation error.
print("best validation depth:", best_depth_a2) # inspect selected capacity.
assert best_depth_a2 == 1

▶ What you'll see: depth 1 wins this validation set.

In [ ]:
plt.figure(figsize=(4.8, 3)) # create a depth-sweep plot.
plt.plot(depths_a2, train_err_a2, marker="o", label="train") # plot training error.
plt.plot(depths_a2, val_err_a2, marker="s", label="validation") # plot validation error.
plt.axvline(best_depth_a2, color="red", linestyle="--", label="selected") # mark best depth.
plt.xlabel("max depth"); plt.ylabel("error rate"); plt.title("Advanced 2: depth regularization"); plt.legend(); plt.show() # display sweep.

▶ What you'll see: a validation-selected depth avoids the overfit outlier split.

👀 Takeaway: `max_depth` trades bias for variance and should be selected with held-out data.

### Advanced 3 — Cost-complexity pruning path

**Goal.** Compute $R(T)+\alpha |T|$ for candidate subtree sizes, because cost-complexity pruning formalizes the price of extra leaves. We build it in 3 steps.

In [ ]:
leaf_counts_a3 = np.array([1, 2, 3]) # candidate subtree sizes: root leaf, stump, deeper tree.
train_risk_a3 = np.array([0.500, 0.125, 0.000]) # toy training classification errors.
alpha_a3 = 0.200 # complexity cost per leaf in this pruning-path demo.
print("leaf counts:", leaf_counts_a3) # inspect model sizes.

▶ What you'll see: larger subtrees have more leaves and lower training risk.

In [ ]:
scores_a3 = train_risk_a3 + alpha_a3 * leaf_counts_a3 # cost-complexity objective.
best_a3 = int(np.argmin(scores_a3)) # select lowest penalized score.
print("penalized scores:", np.round(scores_a3, 3)) # inspect risk plus cost.
print("selected leaves:", leaf_counts_a3[best_a3]) # inspect selected subtree size.
assert leaf_counts_a3[best_a3] == 2

▶ What you'll see: the stump wins because the third leaf's training gain is not worth its added cost.

In [ ]:
plt.figure(figsize=(4.8, 3)) # create a pruning-path plot.
plt.plot(leaf_counts_a3, train_risk_a3, marker="o", label="train risk") # show raw fit.
plt.plot(leaf_counts_a3, scores_a3, marker="s", label="risk + α|T|") # show penalized fit.
plt.axvline(leaf_counts_a3[best_a3], color="red", linestyle="--") # mark selected subtree.
plt.xlabel("number of leaves"); plt.ylabel("score"); plt.title("Advanced 3: pruning objective"); plt.legend(); plt.show() # display path.

▶ What you'll see: raw risk keeps falling, but the penalized score has a minimum at the middle-sized tree.

👀 Takeaway: pruning asks whether each extra leaf earns enough error reduction to pay its complexity cost.

### Advanced 4 — Classification versus regression tree leaves

**Goal.** Compare classification impurity with regression variance reduction, because CART can choose splits for labels or numeric targets using the same weighted-reduction idea. We build it in 3 steps.

In [ ]:
x_a4 = np.array([1., 2., 3., 4., 5., 6.]) # one ordered feature.
y_reg_a4 = np.array([1.0, 1.2, 1.1, 4.0, 4.2, 4.1]) # numeric target with two levels.
thr_a4 = 3.5 # split between the low and high target groups.
left_a4 = x_a4 <= thr_a4 # route examples left.
print("left targets:", y_reg_a4[left_a4], "right targets:", y_reg_a4[~left_a4]) # inspect target groups.

▶ What you'll see: the threshold separates low numeric targets from high numeric targets.

In [ ]:
parent_var_a4 = np.mean((y_reg_a4 - y_reg_a4.mean()) ** 2) # regression-tree impurity: mean squared deviation in parent.
left_var_a4 = np.mean((y_reg_a4[left_a4] - y_reg_a4[left_a4].mean()) ** 2) # left child variance.
right_var_a4 = np.mean((y_reg_a4[~left_a4] - y_reg_a4[~left_a4].mean()) ** 2) # right child variance.
weighted_var_a4 = left_a4.mean() * left_var_a4 + (~left_a4).mean() * right_var_a4 # weighted child variance.
reduction_a4 = parent_var_a4 - weighted_var_a4 # variance reduction.
print("parent var:", round(parent_var_a4, 3), "weighted child var:", round(weighted_var_a4, 3)) # inspect before-after impurity.
print("variance reduction:", round(reduction_a4, 3)) # inspect split gain.
assert round(reduction_a4, 3) == 2.25

▶ What you'll see: the split removes most target variance by grouping similar numeric values.

In [ ]:
pred_reg_a4 = np.where(left_a4, y_reg_a4[left_a4].mean(), y_reg_a4[~left_a4].mean()) # regression leaves predict means.
print("leaf predictions:", np.round(pred_reg_a4, 3)) # inspect piecewise-constant predictions.
plt.figure(figsize=(4.6, 3)) # create regression stump plot.
plt.scatter(x_a4, y_reg_a4, color="black", label="data") # draw targets.
plt.step(x_a4, pred_reg_a4, where="mid", color="teal", label="tree prediction") # draw leaf means.
plt.axvline(thr_a4, color="gray", linestyle="--") # show split.
plt.title("Advanced 4: regression-tree variance reduction"); plt.xlabel("x"); plt.ylabel("target"); plt.legend(); plt.show() # display result.

▶ What you'll see: each side predicts a constant mean, not a class majority.

👀 Takeaway: classification and regression trees share the same split-reduction logic but use different node impurities and leaf predictions.

### Advanced 5 — Show instability from small data changes

**Goal.** Change one label and refit the best split, because single trees can be high-variance models whose structure shifts under small sample changes. We build it in 3 steps.

In [ ]:
x_a5 = np.array([1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5]) # feature values.
y_original_a5 = np.array([0, 0, 0, 1, 1, 1, 1, 0]) # original labels.
y_changed_a5 = y_original_a5.copy() # copy labels for perturbation.
y_changed_a5[-1] = 1 # flip the far-right outlier to match its neighbors.
print("original labels:", y_original_a5) # inspect old labels.
print("changed labels:", y_changed_a5) # inspect perturbed labels.

▶ What you'll see: one label flip removes the outlier.

In [ ]:
def best_threshold_a5(x, y):
    def g(labels):
        c = np.bincount(labels, minlength=2)
        p = c / c.sum()
        return 1 - np.sum(p ** 2)
    parent = g(y)
    ths = (np.unique(x)[:-1] + np.unique(x)[1:]) / 2
    gains = []
    for t in ths:
        L = x <= t
        gains.append(parent - (L.mean() * g(y[L]) + (~L).mean() * g(y[~L])))
    i = int(np.argmax(gains))
    return ths[i], gains[i], np.array(gains)
thr_orig_a5, gain_orig_a5, gains_orig_a5 = best_threshold_a5(x_a5, y_original_a5)
thr_changed_a5, gain_changed_a5, gains_changed_a5 = best_threshold_a5(x_a5, y_changed_a5)
print("original best:", thr_orig_a5, round(gain_orig_a5, 3)) # inspect original split.
print("changed best:", thr_changed_a5, round(gain_changed_a5, 3)) # inspect split after one label flip.
assert round(thr_changed_a5, 3) == 2.25

▶ What you'll see: the best split can become stronger or change its gain after just one label changes.

In [ ]:
ths_a5 = (np.unique(x_a5)[:-1] + np.unique(x_a5)[1:]) / 2 # candidate thresholds.
plt.figure(figsize=(4.8, 3)) # create an instability diagnostic plot.
plt.plot(ths_a5, gains_orig_a5, marker="o", label="original") # plot original gains.
plt.plot(ths_a5, gains_changed_a5, marker="s", label="one label changed") # plot perturbed gains.
plt.xlabel("threshold"); plt.ylabel("Gini gain"); plt.title("Advanced 5: tree split sensitivity"); plt.legend(); plt.show() # display comparison.

▶ What you'll see: the gain curve shifts when one training label changes, illustrating why pruning and ensembles are useful.

👀 Takeaway: single trees are interpretable but can be unstable, so constraints, pruning, and later bagging/random forests reduce variance.